In [1]:
nummers = [1, 2, 3, 4, 5]
print(nummers)
nummers = [n + 1 for n in nummers]
print(nummers)
nummers = [n * 2 for n in nummers]
print(nummers)

[1, 2, 3, 4, 5]
[2, 3, 4, 5, 6]
[4, 6, 8, 10, 12]


In [2]:
# === Generic CSV loader: bestandsnaam -> DataFrame variabele ===
from pathlib import Path
import pandas as pd
import re, keyword
from typing import Dict

# ---- Instellingen
DATA_DIR = Path(".")          # of bijv. Path("./data")
PATTERN  = "*.csv"            # pas aan indien gewenst
NA_VALUES = ["", "NA", "N/A", "na", "null", "Null", "NULL"]

def to_var_name(p: Path) -> str:
    """
    Converteer bestandsnaam naar geldige Python variabelenaam.
    Voorbeeld: 'olympics_raw.csv' -> 'olympics_raw'
    """
    name = p.stem.lower()
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)       # niet-alfanumeriek -> _
    name = re.sub(r"_+", "_", name).strip("_")       # dubbele _ verwijderen
    if not name: 
        name = "df"
    if name[0].isdigit():
        name = f"df_{name}"
    if keyword.iskeyword(name):
        name = f"{name}_"
    return name

def read_csv_safely(path: Path) -> pd.DataFrame:
    """
    Leest CSV met een paar praktische defaults:
    - engine='python' + sep=None -> laat Pandas scheidingsteken 'sniffen'
    - encoding='utf-8-sig' verwerkt BOM correct
    - low_memory=False voor stabielere dtypes
    """
    try:
        return pd.read_csv(
            path,
            sep=None, engine="python",
            encoding="utf-8-sig",
            na_values=NA_VALUES,
            keep_default_na=True,
            low_memory=False
        )
    except Exception as e:
        # Fallback op klassieke komma-separatie
        return pd.read_csv(
            path,
            sep=",",
            encoding="utf-8-sig",
            na_values=NA_VALUES,
            keep_default_na=True,
            low_memory=False
        )

# ---- Inlezen
dfs: Dict[str, pd.DataFrame] = {}
csv_paths = sorted(DATA_DIR.glob(PATTERN))

for p in csv_paths:
    var = to_var_name(p)
    df  = read_csv_safely(p)
    dfs[var] = df
    globals()[var] = df   # maak ook een losse variabele aan
    print(f"Loaded: {var:<25} shape={df.shape}  from '{p.name}'")


Loaded: all_missing_country_codes shape=(1332, 8)  from 'all_missing_country_codes.csv'
Loaded: college_statistics        shape=(777, 19)  from 'college_statistics.csv'
Loaded: country_aliases_manual    shape=(83, 5)  from 'country_aliases_manual.csv'
Loaded: flags_raw                 shape=(193, 5)  from 'flags_raw.csv'
Loaded: gdp_missing_codes         shape=(0, 4)  from 'gdp_missing_codes.csv'
Loaded: gdp_raw                   shape=(266, 68)  from 'gdp_raw.csv'
Loaded: olympics_prepared         shape=(1357, 19)  from 'olympics_prepared.csv'
Loaded: olympics_raw              shape=(20170, 6)  from 'olympics_raw.csv'
Loaded: population_missing_codes  shape=(1332, 4)  from 'population_missing_codes.csv'
Loaded: population_raw            shape=(18944, 4)  from 'population_raw.csv'


In [3]:
#%pip install phonenumbers


import pandas as pd
import pycountry
import phonenumbers

# --- 1️⃣ Functie om telefooncode op te halen via landcode ---
def get_country_calling_code(iso_code):
    try:
        region = phonenumbers.country_code_for_region(iso_code)
        if region:
            return f"+{region}"
        else:
            return None
    except:
        return None

# --- 2️⃣ Bouw lijst met alle landen (via pycountry) ---
countries_data = []
for country in pycountry.countries:
    countries_data.append({
        "country_name": country.name,
        "country_code": country.alpha_2,
        "phone_code": get_country_calling_code(country.alpha_2)
    })

# --- 3️⃣ Maak DataFrame ---
df_countries = pd.DataFrame(countries_data)

# --- 4️⃣ Sorteer en toon ---
df_countries = df_countries.sort_values("country_name").reset_index(drop=True)
print(df_countries.shape)
df_countries.head(10)




(249, 3)


,country_name,country_code,phone_code
0,Afghanistan,AF,+93
1,Albania,AL,+355
2,Algeria,DZ,+213
3,American Samoa,AS,+1
4,Andorra,AD,+376
5,Angola,AO,+244
6,Anguilla,AI,+1
7,Antarctica,AQ,None
8,Antigua and Barbuda,AG,+1
9,Argentina,AR,+54


In [5]:
import pandas as pd
import pycountry
import phonenumbers

# === 1️⃣ Functies voor mapping ===
def get_country_from_input(value):
    """Herken of 'value' een landcode, naam of telefooncode is en geef gestandaardiseerde info terug."""
    if pd.isna(value):
        return None, None, None

    val = str(value).strip().lower()

    # --- probeer ISO2 / ISO3 codes ---
    for country in pycountry.countries:
        if val in [country.alpha_2.lower(), getattr(country, 'alpha_3', '').lower()]:
            return country.name, country.alpha_2, get_phone_code(country.alpha_2)

    # --- probeer naam match (ongevoelig voor case of accenten) ---
    for country in pycountry.countries:
        if val in country.name.lower():
            return country.name, country.alpha_2, get_phone_code(country.alpha_2)

    # --- probeer telefooncode (zoals +31 of 31 of 0031) ---
    val_digits = val.replace("+", "").replace("00", "")
    try:
        val_int = int(val_digits)
        region = phonenumbers.region_code_for_country_code(val_int)
        if region:
            c = pycountry.countries.get(alpha_2=region)
            return c.name, c.alpha_2, f"+{val_int}"
    except:
        pass

    return None, None, None

def get_phone_code(alpha_2):
    """Haal telefooncode op voor een ISO2 code."""
    try:
        code = phonenumbers.country_code_for_region(alpha_2)
        return f"+{code}" if code else None
    except:
        return None


# === 2️⃣ Voorbeeld DataFrame met gemixte input ===
df = pd.DataFrame({
    "country_input": ["NL", "Germany", "turkiye", "90", "United States", "FR", None]
})

# === 3️⃣ Toepassen van de functie ===
df[["country_name", "country_code", "phone_code"]] = df["country_input"].apply(
    lambda x: pd.Series(get_country_from_input(x))
)

print(df)
print(df.shape)

   country_input                          country_name country_code phone_code
0             NL                           Netherlands           NL        +31
1        Germany                               Germany           DE        +49
2        turkiye                                  None         None       None
3             90                               Türkiye           TR        +90
4  United States  United States Minor Outlying Islands           UM       None
5             FR                                France           FR        +33
6           None                                  None         None       None
(7, 4)


In [8]:
print("flag ",flags_raw)

flag                 name  flag_nr_colors flag_mainhue flag_topleft_color  \
0       Afghanistan               5        green              black   
1           Albania               3          red                red   
2           Algeria               3        green              green   
3    American-Samoa               5         blue               blue   
4           Andorra               3         gold               blue   
..              ...             ...          ...                ...   
188   Western-Samoa               3          red               blue   
189      Yugoslavia               4          red               blue   
190           Zaire               4        green              green   
191          Zambia               4        green              green   
192        Zimbabwe               5        green              green   

    flag_botright_color  
0                 green  
1                   red  
2                 white  
3                   red  
4          

In [15]:
import pandas as pd
import pycountry
import phonenumbers

# --- Functie om telefooncode op te halen ---
def get_phone_code(alpha_2):
    try:
        code = phonenumbers.country_code_for_region(alpha_2)
        return f"+{code}" if code else None
    except:
        return None

# --- Functie om willekeurige landnaam of code te standaardiseren ---
def standardize_country(value):
    if pd.isna(value):
        return None, None, None

    val = str(value).strip().lower().replace("-", " ")

    # 1) Probeer directe ISO2/ISO3 match
    for country in pycountry.countries:
        if val in [country.alpha_2.lower(), getattr(country, 'alpha_3', '').lower()]:
            return country.name, country.alpha_2, get_phone_code(country.alpha_2)

    # 2) Probeer naam (flexibele match, hoofdletters en accenten negeren)
    for country in pycountry.countries:
        if val in country.name.lower():
            return country.name, country.alpha_2, get_phone_code(country.alpha_2)

    # 3) Probeer telefooncode (zoals “+31” of “31” of “0031”)
    val_digits = val.replace("+", "").replace("00", "")
    if val_digits.isdigit():
        try:
            region = phonenumbers.region_code_for_country_code(int(val_digits))
            if region:
                c = pycountry.countries.get(alpha_2=region)
                return c.name, c.alpha_2, f"+{val_digits}"
        except:
            pass

    return None, None, None


# === 4️⃣ Toepassen op jouw flag_raw DataFrame ===
# (vervang hier door je eigen DataFrame)
# flag_raw = pd.read_csv("flag_raw.csv")

flags_raw[["country_name_std", "country_code_std", "phone_code"]] = flags_raw["name"].apply(
    lambda x: pd.Series(standardize_country(x))
)

flags_raw[flags_raw["name"] == "USSR"]
print ("flag ",flags_raw[flags_raw["name"] == "USSR"] )  

# === 5️⃣ Resultaat bekijken ===
#flags_raw[["name", "country_name_std", "country_code_std", "phone_code"]].head(15)


flag       name  flag_nr_colors flag_mainhue flag_topleft_color flag_botright_color  \
183  USSR               2          red                red                 red   

    country_name_std country_code_std phone_code  
183             None             None       None  


In [39]:
import pandas as pd
import pycountry
import phonenumbers
import unicodedata
import re

# --- Telefooncode ophalen
def get_phone_code(alpha2):
    try:
        c = phonenumbers.country_code_for_region(alpha2)
        return f"+{c}" if c else None
    except:
        return None

# --- Normaliseren van vrije tekst
def _norm(s: str) -> str:
    s = s.strip().casefold()
    s = s.replace("-", " ").replace("_", " ")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))  # accenten weg
    s = re.sub(r"\s+", " ", s)
    return s

# --- Alias mapping (historisch / synoniemen → ISO2)
ALIAS_TO_ISO2 = {
    # Historische landen
    "ussr": "RU",                  # Sovjet-Unie → Russia
    "soviet union": "RU",
    "zaire": "CD",                 # → Congo (DRC)
    "yugoslavia": "RS",            # pragmatisch → Serbia (kan ook uiteen naar meerdere)
    "czechoslovakia": "CZ",        # pragmatisch → Czechia (CZ) (alternatief SK)
    "burma": "MM",                 # → Myanmar
    "western samoa": "WS",         # → Samoa
    "east timor": "TL",            # → Timor-Leste
    "swaziland": "SZ",             # → Eswatini
    "cape verde": "CV",            # → Cabo Verde
    "macedonia": "MK",             # → North Macedonia
    "ivory coast": "CI",           # → Côte d’Ivoire
    "congo kinshasa": "CD",        # DRC
    "democratic republic of congo": "CD",
    "drc": "CD",
    "congo brazzaville": "CG",     # Republic of the Congo
    "republic of congo": "CG",
    # Veelgebruikte synoniemen/afkortingen
    "uk": "GB",
    "great britain": "GB",
    "united states of america": "US",
    "usa": "US",
    "south korea": "KR",
    "north korea": "KP",
    "czechia": "CZ",
    "laos": "LA",
    "vatican": "VA",
    "palestine": "PS",
    "bolivia": "BO",
    "moldova": "MD",
    "tanzania": "TZ",
    "bahamas": "BS",
    "gambia": "GM",
    "brunei": "BN",
    "syria": "SY",
    "antigua barbuda": "AG",
    "antigua-barbuda": "AG",    
    "Antigua-Barbuda": "AG"
}

# --- Indexen opbouwen voor snelle match
_name_to_iso2 = {}

for c in pycountry.countries:
    iso2 = c.alpha_2
    # hoofdnaam
    _name_to_iso2[_norm(c.name)] = iso2
    # optional fields als ze bestaan
    for attr in ("official_name", "common_name"):
        if hasattr(c, attr):
            _name_to_iso2[_norm(getattr(c, attr))] = iso2

def standardize_country(value):
    """
    Retourneert (country_name, country_code_iso2, phone_code) of (None, None, None)
    Herkent ISO2, ISO3, naam, en aliassen (incl. USSR).
    """
    if pd.isna(value):
        return None, None, None

    raw = str(value)
    val = _norm(raw)

    # 0) Alias/handmatige mapping eerst (dekt USSR)
    if val in ALIAS_TO_ISO2:
        iso2 = ALIAS_TO_ISO2[val]
        c = pycountry.countries.get(alpha_2=iso2)
        return c.name, iso2, get_phone_code(iso2)

    # 1) ISO2/ISO3 directe match
    for c in pycountry.countries:
        if val == c.alpha_2.casefold():
            return c.name, c.alpha_2, get_phone_code(c.alpha_2)
        if hasattr(c, "alpha_3") and val == c.alpha_3.casefold():
            return c.name, c.alpha_2, get_phone_code(c.alpha_2)

    # 2) Naam-match via index (naam / official / common)
    if val in _name_to_iso2:
        iso2 = _name_to_iso2[val]
        c = pycountry.countries.get(alpha_2=iso2)
        return c.name, iso2, get_phone_code(iso2)

    # 3) Bevat-relatie (bijv. "republic of korea" → KR)
    #    (voorzichtig toepassen: kies eerste beste match)
    for key, iso2 in _name_to_iso2.items():
        if val in key:
            c = pycountry.countries.get(alpha_2=iso2)
            return c.name, iso2, get_phone_code(iso2)

    # 4) Telefooncode (zoals +31 / 31 / 0031)
    val_digits = val.replace("+", "")
    if val_digits.startswith("00"):
        val_digits = val_digits[2:]
    if val_digits.isdigit():
        try:
            region = phonenumbers.region_code_for_country_code(int(val_digits))
            if region:
                c = pycountry.countries.get(alpha_2=region)
                return c.name, c.alpha_2, f"+{int(val_digits)}"
        except:
            pass

    return None, None, None


In [40]:
# === flags_std maken (kopie + gestandaardiseerde kolommen toevoegen) ===
flags_std = flags_raw.copy()

# Voeg de gestandaardiseerde waarden toe
flags_std[["country_name_std", "country_code_std", "phone_code"]] = (
    flags_std["name"].apply(lambda x: pd.Series(standardize_country(x)))
)

# Resultaat bekijken
flags_std.head(10)

problem_names = flags_std.loc[flags_std["country_code_std"].isna(), "name"].unique()
print(problem_names)


['British-Virgin-Isles' 'Cape-Verde-Islands' 'Comorro-Islands' 'Faeroes'
 'Falklands-Malvinas' 'Germany-DDR' 'Germany-FRG' 'Kampuchea' 'Malagasy'
 'Maldive-Islands' 'Marianas' 'Netherlands-Antilles' 'North-Yemen'
 'Parguay' 'Soloman-Islands' 'South-Yemen' 'St-Helena' 'St-Kitts-Nevis'
 'St-Lucia' 'St-Vincent' 'Trinidad-Tobago' 'Turkey' 'Turks-Cocos-Islands'
 'UAE' 'US-Virgin-Isles']


In [41]:
# Selecteer rijen waar de standaardnaam of -code ontbreekt
missing_flags = flags_std[
    flags_std["country_name_std"].isna() | flags_std["country_code_std"].isna()
]

# Toon alle rijen volledig (niet afgekapt)
pd.set_option("display.max_rows", None)
print(missing_flags[["name", "country_name_std", "country_code_std", "phone_code"]])
pd.reset_option("display.max_rows")

missing_names = missing_flags["name"].unique().tolist()
print(missing_names)



                     name country_name_std country_code_std phone_code
23   British-Virgin-Isles             None             None       None
31     Cape-Verde-Islands             None             None       None
38        Comorro-Islands             None             None       None
54                Faeroes             None             None       None
55     Falklands-Malvinas             None             None       None
63            Germany-DDR             None             None       None
64            Germany-FRG             None             None       None
91              Kampuchea             None             None       None
102              Malagasy             None             None       None
105       Maldive-Islands             None             None       None
108              Marianas             None             None       None
121  Netherlands-Antilles             None             None       None
128           North-Yemen             None             None       None
134   

In [4]:
# === 0) Helpers ==============================================================
import re, unicodedata
import pandas as pd, pycountry, phonenumbers

def _norm(s: str) -> str:
    s = s.strip().casefold()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.replace("_", " ")
    s = re.sub(r"[-/\.]", " ", s)      # streepjes/slashes/puntjes weg
    s = re.sub(r"\s+", " ", s)
    return s

def get_phone_code(alpha2):
    try:
        c = phonenumbers.country_code_for_region(alpha2)
        return f"+{c}" if c else None
    except:
        return None

# index: naam -> iso2 (inclusief official/common names)
_name_to_iso2 = {}
for c in pycountry.countries:
    _name_to_iso2[_norm(c.name)] = c.alpha_2
    for attr in ("official_name", "common_name"):
        if hasattr(c, attr):
            _name_to_iso2[_norm(getattr(c, attr))] = c.alpha_2

# === 1) Handmatige uitzonderingen / aliassen ================================
ALIAS_TO_ISO2 = {
    # ---- Jouw concrete lijst (exact gedekt) ----
    "british virgin isles": "VG",
    "cape verde islands": "CV",
    "comorro islands": "KM",
    "faeroes": "FO",
    "falklands malvinas": "FK",
    "germany ddr": "DE",
    "germany frg": "DE",
    "kampuchea": "KH",
    "malagasy": "MG",
    "maldive islands": "MV",
    "marianas": "MP",                   # Northern Mariana Islands
    "netherlands antilles": "CW",       # pragmatisch naar Curaçao
    "north yemen": "YE",
    "parguay": "PY",                    # typfout → Paraguay
    "soloman islands": "SB",            # typfout → Solomon Islands
    "south yemen": "YE",
    "st helena": "SH",
    "st kitts nevis": "KN",
    "st lucia": "LC",
    "st vincent": "VC",
    "trinidad tobago": "TT",
    "turkey": "TR",                     # → Türkiye
    "turks cocos islands": "TC",        # Turks and Caicos Islands
    "uae": "AE",
    "us virgin isles": "VI",

    # ---- Historische staten / hernoemingen ----
    "ussr": "RU",
    "soviet union": "RU",
    "zaire": "CD",
    "yugoslavia": "RS",                 # pragmatisch
    "czechoslovakia": "CZ",             # pragmatisch
    "burma": "MM",                      # → Myanmar
    "western samoa": "WS",              # → Samoa
    "swaziland": "SZ",                  # → Eswatini
    "cape verde": "CV",                 # → Cabo Verde
    "macedonia": "MK",                  # → North Macedonia
    "fyrom": "MK",
    "east timor": "TL",                 # → Timor-Leste
    "ivory coast": "CI",                # → Côte d’Ivoire
    "lao pdr": "LA",
    "moldavia": "MD",                   # → Moldova
    "byelorussia": "BY",                # → Belarus
    "malaya": "MY",                     # → Malaysia
    "ceylon": "LK",                     # → Sri Lanka
    "siam": "TH",                       # → Thailand
    "rhodesia": "ZW",                   # → Zimbabwe
    "british honduras": "BZ",           # → Belize
    "upper volta": "BF",                # → Burkina Faso
    "dahomey": "BJ",                    # → Benin
    "persia": "IR",                     # historisch → Iran
    "kampuchea": "KH",                  # (dubbel voor zekerheid)

    # ---- Ambigue/varianten (Congo, UK/US, Korea enz.) ----
    "congo kinshasa": "CD",
    "democratic republic of congo": "CD",
    "drc": "CD",
    "congo brazzaville": "CG",
    "republic of congo": "CG",
    "the congo": "CG",
    "uk": "GB",
    "u k": "GB",
    "great britain": "GB",
    "united states of america": "US",
    "usa": "US",
    "u s a": "US",
    "south korea": "KR",
    "republic of korea": "KR",
    "north korea": "KP",
    "democratic people s republic of korea": "KP",

    # ---- Overzeese gebieden / SAR / Caraïben varianten ----
    "hong kong china": "HK",
    "hong kong sar": "HK",
    "macao china": "MO",
    "macao sar": "MO",
    "curacao": "CW",
    "curaçao": "CW",
    "st maarten": "SX",
    "st. maarten": "SX",
    "saint martin (dutch part)": "SX",
    "saint martin (french part)": "MF",
    "bonaire": "BQ",
    "sint eustatius": "BQ",
    "saba": "BQ",
    "reunion": "RE",
    "tahiti": "PF",                     # Frans-Polynesië
    "greenland": "GL",
    "faroe islands": "FO",              # zekerheid/duplicaat
    "falkland islands": "FK",           # zekerheid/duplicaat

    # ---- Saint-varianten zonder punt & ampersands ----
    "saint helena": "SH",
    "saint kitts and nevis": "KN",
    "saint lucia": "LC",
    "saint vincent and the grenadines": "VC",
    "trinidad and tobago": "TT",
    "antigua and barbuda": "AG",
    "st kitts and nevis": "KN",
    "st vincent and the grenadines": "VC",
    "st barth": "BL",                   # Saint Barthélemy
    "st barthelemy": "BL",
    "st barthelemy": "BL",
}

# === 2) Auto-varianten opbouwen (spaar je veel handwerk) ====================
def expand_alias_variants(alias_map: dict) -> dict:
    extra = {}
    saint_patterns = [
        (r"\bst[\.]?\b", "saint"),
        (r"\bste[\.]?\b", "sainte"),
    ]
    for k, v in alias_map.items():
        base = _norm(k)
        variations = set([base])

        # Varianten met/zonder streepjes/puntjes (al deels in _norm)
        variations.add(base.replace(" ", ""))   # compacte vorm (soms handig)
        variations.add(base.replace(" and ", " & "))
        variations.add(base.replace(" & ", " and "))

        # Saint/Sainte normaliseren
        tmp = base
        for pat, rep in saint_patterns:
            tmp = re.sub(pat, rep, tmp)
        variations.add(tmp)

        # St. → Saint, St- → Saint, St_ → Saint, etc.
        variations.add(re.sub(r"\bst[\. -_]\b", "saint ", base))

        # Voeg alle varianten toe als sleutel
        for var in variations:
            if var not in alias_map and var not in extra:
                extra[var] = v
    out = {**alias_map, **extra}
    return out

ALIAS_TO_ISO2 = expand_alias_variants(ALIAS_TO_ISO2)

# === 3) Standaardiseer-functie (gebruikt zowel pycountry als alias) =========
def standardize_country(value):
    if pd.isna(value): 
        return None, None, None
    raw = str(value)
    val = _norm(raw)

    # 1) alias eerst
    if val in ALIAS_TO_ISO2:
        iso2 = ALIAS_TO_ISO2[val]
        c = pycountry.countries.get(alpha_2=iso2)
        return c.name, iso2, get_phone_code(iso2)

    # 2) directe ISO2/ISO3
    for ctry in pycountry.countries:
        if val == ctry.alpha_2.casefold():
            return ctry.name, ctry.alpha_2, get_phone_code(ctry.alpha_2)
        if hasattr(ctry, "alpha_3") and val == ctry.alpha_3.casefold():
            return ctry.name, ctry.alpha_2, get_phone_code(ctry.alpha_2)

    # 3) exacte naam
    if val in _name_to_iso2:
        iso2 = _name_to_iso2[val]
        c = pycountry.countries.get(alpha_2=iso2)
        return c.name, iso2, get_phone_code(iso2)

    # 4) contains-match (laatste redmiddel; kan ‘false-positives’ geven)
    for key, iso2 in _name_to_iso2.items():
        if val in key:
            c = pycountry.countries.get(alpha_2=iso2)
            return c.name, iso2, get_phone_code(iso2)

    # 5) telefooncode
    digits = val.replace("+", "")
    if digits.startswith("00"):
        digits = digits[2:]
    if digits.isdigit():
        try:
            region = phonenumbers.region_code_for_country_code(int(digits))
            if region:
                c = pycountry.countries.get(alpha_2=region)
                return c.name, c.alpha_2, f"+{int(digits)}"
        except:
            pass

    return None, None, None




In [5]:
# === 4) Toepassen zonder flag_raw te overschrijven ==========================
flags_std = flags_raw.copy()
flags_std[["country_name_std", "country_code_std", "phone_code"]] = (
    flags_std["name"].apply(lambda x: pd.Series(standardize_country(x)))
)
flags_std["matched"] = flags_std["country_code_std"].notna()

# === 5) Alle ‘uitzonderingen’ (nog onopgelost) tonen ========================
not_matched = flags_std.loc[~flags_std["matched"], "name"].dropna().unique()
print("Nog te mappen uitzonderingen:", not_matched)

Nog te mappen uitzonderingen: ['Antigua-Barbuda']


In [ ]:
import re, unicodedata
import pycountry, phonenumbers, pandas as pd

# --- Normalisatie: hoofdletters, accenten, spaties, streepjes negeren ---
def _norm(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip().casefold()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9]", "", s)  # verwijder ALLES behalve letters/cijfers
    return s

# --- Telefooncode helper ---
def get_phone_code(alpha2):
    try:
        c = phonenumbers.country_code_for_region(alpha2)
        return f"+{c}" if c else None
    except:
        return None

# --- Alias mapping (één vorm per land is genoeg) ---
ALIAS_TO_ISO2 = {
    "antigua barbuda": "AG",
    "cape verde islands": "CV",
    "ussr": "RU",
    "turkey": "TR",
    "yugoslavia": "RS",
    "zaire": "CD",
    "czechoslovakia": "CZ",
    "swaziland": "SZ",
    "macedonia": "MK",
    "burma": "MM",
    "kampuchea": "KH",
    # ... vul hier gewoon één normale versie in per alias
}

# --- Voor snelle lookup: genormaliseerde keys ---
ALIAS_TO_ISO2_NORM = {_norm(k): v for k, v in ALIAS_TO_ISO2.items()}

# --- Pycountry index ---
_name_to_iso2 = {}
for c in pycountry.countries:
    iso2 = c.alpha_2
    _name_to_iso2[_norm(c.name)] = iso2
    for attr in ("official_name", "common_name"):
        if hasattr(c, attr):
            _name_to_iso2[_norm(getattr(c, attr))] = iso2

# --- De hoofd-functie ---
def standardize_country(value):
    if pd.isna(value):
        return None, None, None

    key = _norm(value)

    # 1️⃣ eerst alias
    if key in ALIAS_TO_ISO2_NORM:
        iso2 = ALIAS_TO_ISO2_NORM[key]
        c = pycountry.countries.get(alpha_2=iso2)
        return c.name, iso2, get_phone_code(iso2)

    # 2️⃣ officiële naam
    if key in _name_to_iso2:standardize_flags
        iso2 = _name_to_iso2[key]
        c = pycountry.countries.get(alpha_2=iso2)
        return c.name, iso2, get_phone_code(iso2)

    # 3️⃣ fallback: niets gevonden
    return None, None, None


In [43]:
flags_std = flags_raw.copy()
flags_std[["country_name_std", "country_code_std", "phone_code"]] = (
    flags_std["name"].apply(lambda x: pd.Series(standardize_country(x)))
)
flags_std["matched"] = flags_std["country_code_std"].notna()